# VISCERA — full pipeline on Colab (A6000/A100)
**VIS**ual-**C**oncept **E**ndoscopy **R**epresentation via **A**ttributes — for the RARE2026 PPV@90R challenge.

Runs in the **Colab local workspace** (`/content`, fast SSD): git clone code → copy+extract zip data from Drive → build concept targets → **Stage-1 concept-supervised pretraining** → **fair SSL-vs-concept gate** → (if PASS) **Stage-2 downstream**.

### Drive upload checklist (set `DRIVE_DIR` below to your folder)
- `out/*.zip`  — the ~170k concept JSONs + images (you already pushed these). Each must extract to `out/<dir>/images/` + `out/<dir>/labels/`.
- `dinov2.pth` — the domain-adapted backbone.
- `dataset.zip` — labeled images, extracting to `dataset/train/{neo,ndbe}/` and `dataset/val/{neo,ndbe}/`.

Runtime → Change runtime type → **GPU (A6000/A100)**.

In [ ]:
import torch; print(torch.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install timm==1.0.27 scikit-learn

In [ ]:
# ---- clone the code repo ----
REPO_URL = 'https://github.com/<YOUR_USER>/<YOUR_REPO>.git'   # <-- set this
PRIVATE  = True                                              # private repo? need a token
import os
if PRIVATE:
    from getpass import getpass
    tok = getpass('GitHub token (repo scope): ')
    REPO_URL = REPO_URL.replace('https://', f'https://{tok}@')
%cd /content
!rm -rf rare && git clone $REPO_URL rare
%cd /content/rare
!ls

In [ ]:
# ---- pull + extract data from Drive into the repo workspace (relative paths must match) ----
from google.colab import drive; drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/RARE_LG'   # <-- folder with 1.zip, 2.zip, ..., dinov2.pth, dataset.zip
import glob, os, zipfile, shutil
os.makedirs('out', exist_ok=True)

# backbone
assert os.path.exists(f'{DRIVE_DIR}/dinov2.pth'), f'upload dinov2.pth to {DRIVE_DIR}'
shutil.copy(f'{DRIVE_DIR}/dinov2.pth', 'dinov2.pth'); print('dinov2.pth OK')

def extract_chunk(zpath):
    """Extract one numbered data zip so the result is out/<dir>/images + out/<dir>/labels,
    auto-detecting whether the zip root is images|labels/ , <dir>/ , or out/<dir>/."""
    stem = os.path.splitext(os.path.basename(zpath))[0]          # '1', '2', ...
    with zipfile.ZipFile(zpath) as zf:
        tops = {p.split('/')[0] for p in zf.namelist() if p.strip('/')}
        if 'images' in tops or 'labels' in tops:
            target = f'out/{stem}'        # zip root = images/ labels/   -> out/<stem>/
        elif tops == {'out'}:
            target = '.'                  # zip root = out/<dir>/...      -> ./
        else:
            target = 'out'                # zip root = <dir>/...          -> out/<dir>/
        os.makedirs(target, exist_ok=True)
        zf.extractall(target)
    return stem, target

# numbered data zips (1.zip, 2.zip, ...) directly in DRIVE_DIR; exclude any dataset*.zip
data_zips = [z for z in sorted(glob.glob(f'{DRIVE_DIR}/*.zip')) if 'dataset' not in os.path.basename(z).lower()]
print(f'found {len(data_zips)} data zips:', [os.path.basename(z) for z in data_zips[:5]], '...')
for z in data_zips:
    s, t = extract_chunk(z); print(f'  {os.path.basename(z)} -> {t}')

# labeled images zip -> ./dataset/  (must yield dataset/train/{neo,ndbe}, dataset/val/...)
for z in glob.glob(f'{DRIVE_DIR}/dataset*.zip'):
    print('extracting', os.path.basename(z))
    with zipfile.ZipFile(z) as zf: zf.extractall('.')

# sanity check
nlab = len(glob.glob('out/*/labels'))
print('\nout/<dir>/labels found:', nlab, '| sample json:', (glob.glob('out/*/labels/*.json')[:1] or 'NONE'))
print('train neo:', len(glob.glob('dataset/train/neo/*')), 'ndbe:', len(glob.glob('dataset/train/ndbe/*')))
if nlab == 0 and data_zips:
    print('\n!! no out/<dir>/labels — inspect a zip layout and fix extract_chunk:')
    print('   entries:', zipfile.ZipFile(data_zips[0]).namelist()[:5])

In [ ]:
# ---- build the 35-concept target matrix from extracted out/ (self-contained: unlabeled + labeled from JSONs) ----
!python -m phase3.build_concept_targets --out phase3/cache/concept_targets.npz

In [ ]:
# ---- (re)build the 35-concept target matrix from the extracted out/ (paths match Colab layout) ----
!python -m phase3.build_concept_targets --out phase3/cache/concept_targets.npz

## Stage-1 — concept-supervised pretraining on the FULL corpus (A6000 96GB → big batch)

In [ ]:
# Keep Stage-1 light-ish (SHAPE not overwrite SSL): unfreeze 6, bs 192, ~10 epochs. Tune as needed.
!python -m phase3.pretrain_concept --targets phase3/cache/concept_targets.npz \
    --unfreeze 6 --epochs 10 --bs 192 --lr 1e-4 --grl 1.0 --workers 8 --out concept_encoder.pt

In [ ]:
# labeled features from BOTH backbones over the SAME frames (out/train/images via train_colab.csv)
!python -m phase3.featurize --csv train_colab.csv --out feats_train_ssl.npz     --workers 4
!python -m phase3.featurize --csv train_colab.csv --out feats_train_concept.npz --workers 4 --backbone concept_encoder.pt
!python -m phase3.concept_gate --ssl feats_train_ssl.npz --concept feats_train_concept.npz --C 1.0

In [ ]:
!python -m phase3.featurize --csv dataset/train.csv --out feats_train_ssl.npz     --workers 4
!python -m phase3.featurize --csv dataset/train.csv --out feats_train_concept.npz --workers 4 --backbone concept_encoder.pt
!python -m phase3.concept_gate --ssl feats_train_ssl.npz --concept feats_train_concept.npz --C 1.0

In [ ]:
# hold out center_2 then center_1; compare per-epoch LOCO-val PPV@90R of the two inits (uses out/train images)
for hold in ['center_2','center_1']:
    print('=== concept-init, holdout', hold, '===')
    !python -m phase3.finetune --train-csv train_colab.csv --holdout {hold} --init concept_encoder.pt --unfreeze 6 --epochs 30 --bs 96 --loss bce+rank+pauc --out ft_concept_{hold}.pt
    print('=== ssl-init, holdout', hold, '===')
    !python -m phase3.finetune --train-csv train_colab.csv --holdout {hold}                          --unfreeze 6 --epochs 30 --bs 96 --loss bce+rank+pauc --out ft_ssl_{hold}.pt
import shutil; shutil.copy('concept_encoder.pt', f'{DRIVE_DIR}/concept_encoder.pt'); print('saved encoder to Drive')

In [ ]:
# hold out center_2 then center_1; compare the per-epoch LOCO-val PPV@90R of the two inits
for hold in ['center_2','center_1']:
    print('=== concept-init, holdout', hold, '==='); 
    !python -m phase3.finetune --holdout {hold} --init concept_encoder.pt --unfreeze 6 --epochs 30 --bs 96 --loss bce+rank+pauc --out ft_concept_{hold}.pt
    print('=== ssl-init, holdout', hold, '==='); 
    !python -m phase3.finetune --holdout {hold}                          --unfreeze 6 --epochs 30 --bs 96 --loss bce+rank+pauc --out ft_ssl_{hold}.pt
# save encoder back to Drive
import shutil; shutil.copy('concept_encoder.pt', f'{DRIVE_DIR}/concept_encoder.pt'); print('saved encoder to Drive')